<a href="https://colab.research.google.com/github/crialejo24/DOWNSCALING/blob/main/Dataset_S2_LS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import ee
import geemap
import pandas as pd
import numpy as np
import random
import cv2
from google.colab.patches import cv2_imshow
import os

#1. USE COLAB WITH A GOOGLE PERSONAL ACCOUNT (NOT @elpoli.edu.co)
#2. SIGN IN TO https://developers.google.com/earth-engine
#3. CREATE A PROJECT, IN THIS CASE I NAMED IT "rubenchov". USE YOUR OWN.

# Drive
drive.mount('/content/drive')

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
#ee.Initialize(project='vamos-486322')  #CRISTIAN
ee.Initialize(project='andresc-488619') #GUSTAVO

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###FUNCIONES

In [ ]:
# =========================================================
# NUBES SENTINEL
# =========================================================
def cloud_percentage_s2(image, roi):
    # ---------- SCL ----------
    scl = image.select('SCL')
    scl_clouds = (
        scl.eq(3)   # shadow
        .Or(scl.eq(8))
        .Or(scl.eq(9))
        .Or(scl.eq(10))
    )

    #.Or(scl.eq(7) Clouds Low Probability / Unclassified
    #.Or(scl.eq(11) Snow / Ice

    # ---------- QA60 ----------
    qa60 = image.select('QA60')
    qa_cloud = qa60.bitwiseAnd(1 << 10).neq(0)
    qa_cirrus = qa60.bitwiseAnd(1 << 11).neq(0)
    qa_clouds = qa_cloud.Or(qa_cirrus)

    # ---------- combinación ----------
    clouds = scl_clouds.Or(qa_clouds)

    stats = clouds.reduceRegion(
        reducer=ee.Reducer.sum().combine(
            reducer2=ee.Reducer.count(),
            sharedInputs=True
        ),
        geometry=roi,
        scale=20,
        maxPixels=1e9
    )

    cloud_pixels = ee.Number(stats.get('SCL_sum'))
    total_pixels = ee.Number(stats.get('SCL_count'))
    cloud_pct = cloud_pixels.divide(total_pixels).multiply(100)

    return image.set('CLOUD_PCT_ROI', cloud_pct)


# =========================================================
# AGUA SENTINEL (NDWI)
# =========================================================
def water_percentage_s2(image, roi, threshold=0.00):

    green = image.select('B3').multiply(1e-4)
    nir   = image.select('B8').multiply(1e-4)

    ndwi = green.subtract(nir).divide(green.add(nir)).rename('NDWI')
    water_mask = ndwi.gt(threshold).rename('water')

    stats = water_mask.reduceRegion(
        reducer=ee.Reducer.sum().combine(
            reducer2=ee.Reducer.count(),
            sharedInputs=True
        ),
        geometry=roi,
        scale=10,
        maxPixels=1e9
    )

    water_pixels = ee.Number(stats.get('water_sum'))
    total_pixels = ee.Number(stats.get('water_count'))
    water_pct = water_pixels.divide(total_pixels).multiply(100)

    ndwi_mean = ndwi.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,
        maxPixels=1e9
    ).get('NDWI')

    return image.set({
        'WATER_PCT_ROI': water_pct,
        'NDWI_MEAN_ROI': ndwi_mean
    })


# =========================================================
# NUBES LANDSAT
# =========================================================
def cloud_percentage_landsat(image, roi):

    qa = image.select('QA_PIXEL')

    dilated = qa.bitwiseAnd(1 << 1).neq(0)
    cirrus = qa.bitwiseAnd(1 << 2).neq(0)
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    #snow = qa.bitwiseAnd(1 << 5).neq(0)

    clouds = dilated.Or(cirrus).Or(cloud).Or(shadow)#.Or(snow)

    stats = clouds.reduceRegion(
        reducer=ee.Reducer.sum().combine(
            reducer2=ee.Reducer.count(),
            sharedInputs=True
        ),
        geometry=roi,
        scale=30,
        maxPixels=1e9
    )

    cloud_pixels = ee.Number(stats.get('QA_PIXEL_sum'))
    total_pixels = ee.Number(stats.get('QA_PIXEL_count'))
    cloud_pct = cloud_pixels.divide(total_pixels).multiply(100)

    return image.set('CLOUD_PCT_ROI', cloud_pct)


# =========================================================
# VARIABILIDAD SENTINEL (CV RGB %)
# =========================================================
def variability_s2(image, roi):

    rgb = image.select(['B4', 'B3', 'B2']).multiply(1e-4)

    stats = rgb.reduceRegion(
        reducer=ee.Reducer.mean().combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        ),
        geometry=roi,
        scale=10,
        maxPixels=1e9
    )

    mean_r = ee.Number(stats.get('B4_mean'))
    mean_g = ee.Number(stats.get('B3_mean'))
    mean_b = ee.Number(stats.get('B2_mean'))

    std_r = ee.Number(stats.get('B4_stdDev'))
    std_g = ee.Number(stats.get('B3_stdDev'))
    std_b = ee.Number(stats.get('B2_stdDev'))

    mean_rgb = mean_r.add(mean_g).add(mean_b).divide(3)
    std_rgb = std_r.add(std_g).add(std_b).divide(3)

    cv_rgb = ee.Algorithms.If(
        mean_rgb.neq(0),
        std_rgb.divide(mean_rgb).multiply(100),
        ee.Number(0)
    )

    return image.set('STD_RGB_ROI', cv_rgb)


# =========================================================
# NORMALIZACIÓN ROBUSTA
# =========================================================
def normalize_if_needed(img):
    img = img.astype(np.float32)
    max_val = np.nanmax(img)
    if max_val > 1.5:
        img = img / 255.0
    return img



def get_temporal_pair_rgb(
        start,
        end,
        latitude,
        longitude,
        dim=512,
        max_cloud_pct=5,
        max_water_pct=50,
        day_tolerance=5):

    px = 10
    dimensions = 1
    d = 0

    while dimensions < dim:

        roi = ee.Geometry.Point([longitude, latitude]).buffer((dim + d) * px / 2).bounds()
        coords = roi.getInfo()['coordinates'][0]

        lon1, lat1 = coords[3]
        lon2, lat2 = coords[1]

        p11 = ee.Geometry.Point(lon1, lat1)
        p12 = ee.Geometry.Point(lon1, lat2)
        p21 = ee.Geometry.Point(lon2, lat1)
        p22 = ee.Geometry.Point(lon2, lat2)

        s2_col = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(p11).filterBounds(p12)
            .filterBounds(p21).filterBounds(p22)
            .filterDate(start, end)
            .map(lambda img: cloud_percentage_s2(img, roi))
            .map(lambda img: water_percentage_s2(img, roi))
            .map(lambda img: variability_s2(img, roi))
            .filter(ee.Filter.lte('CLOUD_PCT_ROI', max_cloud_pct))
            .filter(ee.Filter.lte('WATER_PCT_ROI', max_water_pct))
            .filter(ee.Filter.gte('STD_RGB_ROI', 20))
        )

        roi = ee.Geometry.Rectangle(lon1, lat1, lon2, lat2)

        if s2_col.size().getInfo() == 0:
            print("❌ No hay Sentinel válida")
            return None

        first_img = (
            s2_col.first()
            .select('B2')
            .clip(roi)
            .reproject(crs='EPSG:4326', scale=10)
        )

        info = first_img.getInfo()
        dimensions = min(info['bands'][0]['dimensions'])
        s2_dim = info['bands'][0]['dimensions']

        d += 1

    # ================= LANDSAT =================
    landsat_col = (
        ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
        .filterBounds(p11).filterBounds(p12)
        .filterBounds(p21).filterBounds(p22)
        .filterDate(start, end)
        .map(lambda img: cloud_percentage_landsat(img, roi))
        .filter(ee.Filter.lte('CLOUD_PCT_ROI', max_cloud_pct))
    )

    if landsat_col.size().getInfo() == 0:
        print("❌ No hay Landsat válida")
        return None

    # ================= MATCH TEMPORAL =================
    s2_list = s2_col.toList(s2_col.size())
    landsat_list = landsat_col.toList(landsat_col.size())

    best_pair = None
    min_diff = 9999

    for i in range(s2_col.size().getInfo()):
        s2_img_tmp = ee.Image(s2_list.get(i))
        s2_date_tmp = ee.Date(s2_img_tmp.get('system:time_start'))

        for j in range(landsat_col.size().getInfo()):
            ls_img_tmp = ee.Image(landsat_list.get(j))
            ls_date_tmp = ee.Date(ls_img_tmp.get('system:time_start'))

            diff = abs(s2_date_tmp.difference(ls_date_tmp, 'day').getInfo())

            if diff < min_diff:
                min_diff = diff
                best_pair = (s2_img_tmp, ls_img_tmp)
                best_s2_date = s2_date_tmp
                best_ls_date = ls_date_tmp

    if best_pair is None or min_diff > day_tolerance:
        print("❌ No existe par dentro de la tolerancia temporal")
        return None

    print(f"✅ Par encontrado con diferencia de {min_diff} días")

    # ================= IMÁGENES =================
    s2_img, ls_img = best_pair

    s2_image = (
        s2_img.select([
            'B1','B2','B3','B4','B5','B6','B7',
            'B8','B8A','B11','B12'
        ])
        .clip(roi)
        .reproject(crs='EPSG:4326', scale=10)
    )

    landsat_image = (
        ls_img.select([
            'SR_B1','SR_B2','SR_B3','SR_B4',
            'SR_B5','SR_B6','SR_B7'
        ])
        .clip(roi)
        .reproject(crs='EPSG:4326', scale=30)
    )

    # ===== dimensiones Landsat =====
    first_ls = landsat_image.select('SR_B2')
    ls_info = first_ls.getInfo()
    ls_dim = ls_info['bands'][0]['dimensions']

    # ================= RGB EN REFLECTANCIA REAL =================
    rgb_s2 = s2_image.select(['B4','B3','B2']).multiply(1e-4)

    rgb_landsat = (
        landsat_image
        .select(['SR_B4','SR_B3','SR_B2'])
        .multiply(2.75e-5)
        .add(-0.2)
    )

    s2_date_str = best_s2_date.format('YYYY-MM-dd').getInfo()
    ls_date_str = best_ls_date.format('YYYY-MM-dd').getInfo()

    return {
        "rgb_s2": rgb_s2,
        "rgb_landsat": rgb_landsat,
        "s2_full": s2_image,
        "landsat_full": landsat_image,
        "roi": roi,
        "s2_date": s2_date_str,
        "landsat_date": ls_date_str,
        "s2_dimension": s2_dim,
        "landsat_dimension": ls_dim
    }

import rasterio
from rasterio.transform import from_bounds
import os
import numpy as np
import cv2

def to_uint8_vis(img):
      """Stretch simple para visualización"""
      img = np.clip(img / 0.3, 0, 1)
      return (img * 255).astype(np.uint8)

def to_uint8_rgb(img):

      img = np.clip(img, 0, 1)
      return (img * 255).astype(np.uint8)

import os
import cv2
import numpy as np
import rasterio
import geemap
from rasterio.transform import from_bounds

def save_images(images_dict, nombre, folder, dim=192):
    # ============================
    # Subcarpetas
    # ============================
    subfolder_1="HR_192_norm/"        # HR Sentinel float
    subfolder_2="LR_48_normx4/"       # LR/4 Landsat float
    subfolder_3="FB_HR_192/"          # Full HR
    subfolder_4="FB_LR_48/"           # Full LR
    subfolder_6="HR_192_vis/"         # HR visualizable
    subfolder_7="LR_48_visx4/"        # LR/4 visualizable
    subfolder_8="HR_192_mod/"         # HR para modelo
    subfolder_9="LR_48_modx4/"        # LR/4 para modelo

    # Carpetas LR/3
    subfolder_10="LR_64_normx3/"      # LR/3 Landsat float
    subfolder_11="LR_64_modx3/"       # LR/3 para modelo
    subfolder_12="LR_64_visx3/"       # LR/3 visualizable

    # Crear carpetas
    for f in [subfolder_1, subfolder_2, subfolder_3, subfolder_4,
              subfolder_6, subfolder_7, subfolder_8, subfolder_9,
              subfolder_10, subfolder_11, subfolder_12]:
        os.makedirs(folder+f, exist_ok=True)

    roi = images_dict["roi"]
    bounds = roi.bounds().getInfo()['coordinates'][0]
    xmin, ymin = bounds[0]
    xmax, ymax = bounds[2]

    # ================= SENTINEL HR =================
    rgb_s2_np = geemap.ee_to_numpy(images_dict["rgb_s2"], region=roi, scale=10).astype(np.float32)
    rgb_s2_resized = cv2.resize(rgb_s2_np, (dim, dim), interpolation=cv2.INTER_AREA).astype(np.float32)
    rgb_s2_resized = np.clip(rgb_s2_resized, 0, 1).astype(np.float32)

    transform_s2 = from_bounds(xmin, ymin, xmax, ymax, dim, dim)

    # HR float
    with rasterio.open(os.path.join(folder+subfolder_1, nombre+"_RGB_S2_float.tif"),
                       'w', driver='GTiff', height=dim, width=dim, count=3,
                       dtype='float32', crs='EPSG:4326', transform=transform_s2) as dst:
        for i in range(3):
            dst.write(rgb_s2_resized[:, :, i], i+1)

    # HR mod
    rgb_s2_mod = to_uint8_rgb(rgb_s2_resized)
    with rasterio.open(os.path.join(folder+subfolder_8, nombre+"_RGB_S2_mod.tif"),
                       'w', driver='GTiff', height=dim, width=dim, count=3,
                       dtype='uint8', crs='EPSG:4326', transform=transform_s2) as dst:
        for i in range(3):
            dst.write(rgb_s2_mod[:, :, i], i+1)

    # HR visualizable
    rgb_s2_vis = to_uint8_vis(rgb_s2_resized)
    with rasterio.open(os.path.join(folder+subfolder_6, nombre+"_RGB_S2_vis.tif"),
                       'w', driver='GTiff', height=dim, width=dim, count=3,
                       dtype='uint8', crs='EPSG:4326', transform=transform_s2) as dst:
        for i in range(3):
            dst.write(rgb_s2_vis[:, :, i], i+1)
    print("✅ Guardado HR Sentinel")

    # ================= LANDSAT LR =================
    rgb_ls_np = geemap.ee_to_numpy(images_dict["rgb_landsat"], region=roi, scale=30).astype(np.float32)
    rgb_ls_np = np.clip(rgb_ls_np, 0, 1).astype(np.float32)

    # --- LR/4 ---
    lr4_dim = int(dim/4)
    rgb_ls_lr4 = cv2.resize(rgb_ls_np, (lr4_dim, lr4_dim), interpolation=cv2.INTER_AREA)
    transform_ls_lr4 = from_bounds(xmin, ymin, xmax, ymax, lr4_dim, lr4_dim)

    # LR/4 float
    with rasterio.open(os.path.join(folder+subfolder_2, nombre+"_RGB_LS_float_x4.tif"),
                       'w', driver='GTiff', height=lr4_dim, width=lr4_dim, count=3,
                       dtype='float32', crs='EPSG:4326', transform=transform_ls_lr4) as dst:
        for i in range(3):
            dst.write(rgb_ls_lr4[:, :, i], i+1)

    # LR/4 mod
    rgb_ls_lr4_mod = to_uint8_rgb(rgb_ls_lr4)
    with rasterio.open(os.path.join(folder+subfolder_9, nombre+"_RGB_LS_mod_x4.tif"),
                       'w', driver='GTiff', height=lr4_dim, width=lr4_dim, count=3,
                       dtype='uint8', crs='EPSG:4326', transform=transform_ls_lr4) as dst:
        for i in range(3):
            dst.write(rgb_ls_lr4_mod[:, :, i], i+1)

    # LR/4 visualizable
    rgb_ls_lr4_vis = to_uint8_vis(rgb_ls_lr4)
    with rasterio.open(os.path.join(folder+subfolder_7, nombre+"_RGB_LS_vis_x4.tif"),
                       'w', driver='GTiff', height=lr4_dim, width=lr4_dim, count=3,
                       dtype='uint8', crs='EPSG:4326', transform=transform_ls_lr4) as dst:
        for i in range(3):
            dst.write(rgb_ls_lr4_vis[:, :, i], i+1)
    print("✅ Guardado LR/4 Landsat")

    # --- LR/3 ---
    lr3_dim = int(dim/3)
    rgb_ls_lr3 = cv2.resize(rgb_ls_np, (lr3_dim, lr3_dim), interpolation=cv2.INTER_AREA)
    transform_ls_lr3 = from_bounds(xmin, ymin, xmax, ymax, lr3_dim, lr3_dim)

    # LR/3 float
    with rasterio.open(os.path.join(folder+subfolder_10, nombre+"_RGB_LS_float_x3.tif"),
                       'w', driver='GTiff', height=lr3_dim, width=lr3_dim, count=3,
                       dtype='float32', crs='EPSG:4326', transform=transform_ls_lr3) as dst:
        for i in range(3):
            dst.write(rgb_ls_lr3[:, :, i], i+1)

    # LR/3 mod
    rgb_ls_lr3_mod = to_uint8_rgb(rgb_ls_lr3)
    with rasterio.open(os.path.join(folder+subfolder_11, nombre+"_RGB_LS_mod_x3.tif"),
                       'w', driver='GTiff', height=lr3_dim, width=lr3_dim, count=3,
                       dtype='uint8', crs='EPSG:4326', transform=transform_ls_lr3) as dst:
        for i in range(3):
            dst.write(rgb_ls_lr3_mod[:, :, i], i+1)

    # LR/3 visualizable
    rgb_ls_lr3_vis = to_uint8_vis(rgb_ls_lr3)
    with rasterio.open(os.path.join(folder+subfolder_12, nombre+"_RGB_LS_vis_x3.tif"),
                       'w', driver='GTiff', height=lr3_dim, width=lr3_dim, count=3,
                       dtype='uint8', crs='EPSG:4326', transform=transform_ls_lr3) as dst:
        for i in range(3):
            dst.write(rgb_ls_lr3_vis[:, :, i], i+1)
    print("✅ Guardado LR/3 Landsat")

    # ================= EXPORT FULL =================
    geemap.ee_export_image(
        images_dict["s2_full"],
        filename=os.path.join(folder+subfolder_3, nombre+"_FB_S2.tif"),
        region=roi, scale=10, file_per_band=False
    )
    geemap.ee_export_image(
        images_dict["landsat_full"],
        filename=os.path.join(folder+subfolder_4, nombre+"_FB_LS.tif"),
        region=roi, scale=30, file_per_band=False
    )
    print("🎉 Todo guardado correctamente")


def nombre_random(n=10):
    caracteres = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    nombre = ''.join(random.choice(caracteres) for i in range(n))
    return nombre


###GENERACIÓN DEL DATASET

In [ ]:
import random
import datetime
import csv
import os
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# =====================================
# CONFIGURACIÓN
# =====================================
tipo='train'
#tipo='test'

if tipo=='test':
    base_folder = "/content/drive/MyDrive/DOWNSCALING_2/test_GeoR/"
else:
    base_folder = "/content/drive/MyDrive/DOWNSCALING_2/train_GeoR/"

os.makedirs(base_folder, exist_ok=True)

csv_filename = os.path.join(base_folder, "dataset_metadata_192_48_64.csv")

geolocator = Nominatim(user_agent="dataset_generator")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

TARGET_IMAGES = 10000
S2_RESIZED_DIM = 192
LS_RESIZED_DIM_4 = S2_RESIZED_DIM // 4  # LR/4
LS_RESIZED_DIM_3 = S2_RESIZED_DIM // 3  # LR/3
max_intentos = 20000

# =====================================
# FUNCIÓN FECHA ALEATORIA
# =====================================
def random_month_between(start_year=2022, end_year=2023, end_month_limit=1):
    year = random.randint(start_year, end_year)
    month = random.randint(1, 12) if year != end_year else random.randint(1, end_month_limit)
    start_date = datetime.date(year, month, 1)
    end_date = datetime.date(year + (month == 12), 1 if month == 12 else month + 1, 1) - datetime.timedelta(days=1)
    return str(start_date), str(end_date)

# =====================================
# CONTAR EXISTENTES
# =====================================
existing_names = set()
imagenes_generadas = 0

if os.path.isfile(csv_filename):
    with open(csv_filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader, None)
        for row in reader:
            if len(row) > 0:
                existing_names.add(row[0])
                imagenes_generadas += 1

print(f"📊 Imágenes ya existentes: {imagenes_generadas}")

# =====================================
# CREAR CSV SI NO EXISTE
# =====================================
file_exists = os.path.isfile(csv_filename)

with open(csv_filename, mode='a', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    if not file_exists:
        writer.writerow([
            "filename",
            "latitude",
            "longitude",
            "sentinel_original_dimension",
            "landsat_original_dimension",
            "sentinel_resized_dimension",
            "landsat_resized_LR4_dimension",
            "landsat_resized_LR3_dimension",
            "city",
            "country",
            "sentinel_date",
            "landsat_date"
        ])

    intentos = 0

    while imagenes_generadas < TARGET_IMAGES and intentos < max_intentos:
        intentos += 1
        lat = random.uniform(-30, 30)
        lon = random.uniform(-179, 179)
        start_date, end_date = random_month_between()

        print(f"\nIntento {intentos}")
        print(f"📍 {lat:.4f}, {lon:.4f}")
        print(f"📅 {start_date} → {end_date}")

        result = get_temporal_pair_rgb(
            start=start_date,
            end=end_date,
            latitude=lat,
            longitude=lon,
            dim=S2_RESIZED_DIM,
            max_cloud_pct=1,
            day_tolerance=5
        )

        if result is None:
            print("❌ No válida")
            continue

        # =========================
        # Nombre único
        # =========================
        nombre = nombre_random()
        while nombre in existing_names:
            nombre = nombre_random()

        # =========================
        # Guardar imágenes HR/LR
        # =========================
        save_images(result, nombre, base_folder, dim=S2_RESIZED_DIM)

        # =========================
        # Obtener ciudad y país
        # =========================
        try:
            location = reverse((lat, lon), language='en')
            if location and 'address' in location.raw:
                address = location.raw['address']
                city = address.get('city', address.get('town', address.get('village', 'Unknown')))
                country = address.get('country', 'Unknown')
            else:
                city = "Unknown"
                country = "Unknown"
        except:
            city = "Unknown"
            country = "Unknown"

        # =========================
        # METADATOS
        # =========================
        sentinel_dim_original = result["s2_dimension"]
        landsat_dim_original = result["landsat_dimension"]

        sentinel_dim_resized = (S2_RESIZED_DIM, S2_RESIZED_DIM)
        landsat_dim_resized_LR4 = (LS_RESIZED_DIM_4, LS_RESIZED_DIM_4)
        landsat_dim_resized_LR3 = (LS_RESIZED_DIM_3, LS_RESIZED_DIM_3)

        sentinel_date = result["s2_date"]
        landsat_date = result["landsat_date"]

        # =========================
        # Guardar CSV
        # =========================
        writer.writerow([
            nombre,
            lat,
            lon,
            sentinel_dim_original,
            landsat_dim_original,
            sentinel_dim_resized,
            landsat_dim_resized_LR4,
            landsat_dim_resized_LR3,
            city,
            country,
            sentinel_date,
            landsat_date
        ])

        file.flush()
        existing_names.add(nombre)
        imagenes_generadas += 1

        print(f"✅ Imagen {imagenes_generadas}/{TARGET_IMAGES} guardada")

print("\n=================================")
print(f"Total final: {imagenes_generadas}")
print("Proceso terminado")



📊 Imágenes ya existentes: 3544

Intento 1
📍 6.8153, -46.2047
📅 2022-02-01 → 2022-02-28


NameError: name 'get_temporal_pair_rgb' is not defined

##VERIFICACIÓN DE CORRESPONDENCIA DE IMAGENES ENTRE LAS CARPETAS

In [ ]:
import os
import pandas as pd

# ===============================
# RUTA BASE
# ===============================
tipo='test'
#tipo='train'

if tipo == 'test':
    base = "/content/drive/MyDrive/DOWNSCALING_2/test_GeoR/"
else:
    base = "/content/drive/MyDrive/DOWNSCALING_2/train_GeoR/"

# Ruta del CSV
csv_path = os.path.join(base, "dataset_metadata_192_48_64.csv")  # ajusta el nombre

# ===============================
# LEER CSV
# ===============================
df = pd.read_csv(csv_path)

# Crear columna auxiliar con clave
df['clave'] = df['filename'].str[:10]

claves_csv = set(df['clave'])

print("Claves en CSV:", len(claves_csv))

# ===============================
# CARPETAS
# ===============================
carpetas = [
    "HR_192_norm/",
    "LR_48_normx4/",
    "FB_HR_192/",
    "FB_LR_48/",
    "HR_192_vis/",
    "LR_48_visx4/",
    "HR_192_mod/",
    "LR_48_modx4/",
    "LR_64_normx3/",
    "LR_64_modx3/",
    "LR_64_visx3/"
]

paths = [os.path.join(base, c) for c in carpetas]

# ===============================
# LEER ARCHIVOS
# ===============================
archivos_por_carpeta = {}

for path in paths:
    archivos = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]

    claves = {}
    for a in archivos:
        clave = a[:10]
        claves[clave] = a

    archivos_por_carpeta[path] = claves

# ===============================
# INTERSECCIÓN DE CLAVES
# ===============================
sets = [set(d.keys()) for d in archivos_por_carpeta.values()]
claves_comunes = set.intersection(*sets)

# Intersección con CSV
claves_comunes = claves_comunes.intersection(claves_csv)

print("Imágenes válidas (carpetas + CSV):", len(claves_comunes))

# ===============================
# BORRAR ARCHIVOS SOBRANTES
# ===============================
eliminadas = 0

for path, archivos_dict in archivos_por_carpeta.items():
    for clave, archivo in archivos_dict.items():
        if clave not in claves_comunes:
            ruta = os.path.join(path, archivo)
            os.remove(ruta)
            eliminadas += 1
            print("Eliminado archivo:", ruta)

print("Total archivos eliminados:", eliminadas)

# ===============================
# LIMPIAR CSV
# ===============================
df_filtrado = df[df['clave'].isin(claves_comunes)].copy()

eliminadas_csv = len(df) - len(df_filtrado)

# Opcional: quitar columna auxiliar
df_filtrado = df_filtrado.drop(columns=['clave'])

# Guardar (sobrescribe)
df_filtrado.to_csv(csv_path, index=False)

print("Filas eliminadas del CSV:", eliminadas_csv)
print("CSV alineado correctamente.")

print("Dataset completamente alineado (carpetas + CSV).")

Claves en CSV: 500
Imágenes válidas (carpetas + CSV): 500
Total archivos eliminados: 0
Filas eliminadas del CSV: 0
CSV alineado correctamente.
Dataset completamente alineado (carpetas + CSV).
